<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_48_terraform_for_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🌍 **Module 48 — Terraform for ML Infrastructure (the layer below Kubernetes)** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)


# Module 48 — Terraform for ML Infrastructure

> Helm (M47) configures **inside** a Kubernetes cluster. **Terraform** configures **the cluster itself** — and the VPC, subnets, IAM roles, S3 buckets, RDS instances, GPU node pools, DNS records, secrets stores, and every other cloud thing your ML platform sits on. It's how you turn "click-ops in the AWS console" into a Git-reviewed, reproducible, multi-region infrastructure.
>
> By the end of this module you can read 90% of real ML-platform Terraform code and contribute to it.

> 🟡 We won't run `terraform apply` from this notebook (no cloud creds), but every snippet is production-shaped HCL you can drop into a `main.tf`.

### What you'll cover
1. Why IaC — and why Terraform won
2. The 5 building blocks: provider · resource · data · variable · output
3. **State** — the most-misunderstood concept in Terraform
4. **Modules** — reusable building blocks
5. A real ML platform: VPC + EKS + GPU node pool + S3 + IAM
6. **`terraform init / plan / apply / destroy`** workflow
7. **Workspaces** + remote state for dev / staging / prod
8. Secrets — never put them in `.tf`
9. Terraform vs Pulumi vs CDK vs OpenTofu vs Crossplane
10. Day-2 ops + anti-patterns


## 1 · Why IaC, why Terraform

Without IaC: someone clicks 47 buttons in the AWS console to create your EKS cluster. Two weeks later, nobody remembers exactly what they clicked. The disaster-recovery plan is "we hope".

With IaC: the cluster is **a file in Git**. `terraform apply` recreates it from scratch in 10 minutes. Pull requests review changes. Drift detection catches console-edits.

| Tool | Language | Notes |
|---|---|---|
| **Terraform / OpenTofu** | HCL (declarative) | dominant; OpenTofu = OSS fork after license change |
| **Pulumi** | Python / TS / Go | real programming language; same providers as Terraform |
| **AWS CDK / CDKTF** | TS / Python → CloudFormation / TF | nicer abstractions, less portable |
| **Crossplane** | Kubernetes CRDs | provision cloud from inside the cluster (interesting for k8s shops) |
| **Ansible** | YAML (imperative) | **configures servers**, not provisions infra (M49) |

**Default pick in 2026:** **OpenTofu** for new projects (or Terraform if your org licensed it before mid-2023). Pulumi if your team strongly prefers TypeScript. Always pair with **Atlantis / Spacelift / Terraform Cloud** for safe `apply` from PRs.

## 2 · The 5 building blocks

In [ ]:
terraform_basics = '''# 1) PROVIDER — the SDK for a cloud / SaaS
terraform {
  required_providers {
    aws = { source = "hashicorp/aws", version = "~> 5.60" }
  }
}
provider "aws" {
  region = var.region
}

# 2) VARIABLE — typed input
variable "region"      { type = string,                      default = "us-east-1" }
variable "cluster_name"{ type = string }
variable "tags"        { type = map(string),                 default = { team = "ml" } }

# 3) DATA — read something that already exists
data "aws_caller_identity" "current" {}

# 4) RESOURCE — declare something to be managed
resource "aws_s3_bucket" "models" {
  bucket = "${var.cluster_name}-models"
  tags   = var.tags
}

# 5) OUTPUT — expose values to the caller
output "bucket_arn"   { value = aws_s3_bucket.models.arn }
output "account_id"   { value = data.aws_caller_identity.current.account_id }
'''
print(terraform_basics)

**Reading guide.**
- `provider` = the API client. Each cloud / SaaS has one.
- `resource "aws_s3_bucket" "models"` — *type* `aws_s3_bucket`, *local name* `models`. Refer to it later as `aws_s3_bucket.models`.
- `data` = read-only lookup of something that exists outside Terraform.
- `variable` = typed inputs. Pass at the CLI (`-var`), via `terraform.tfvars`, or env vars.
- `output` = let parent modules read your computed values.


## 3 · State — the central concept

Terraform keeps a **state file** mapping `aws_s3_bucket.models` → the real S3 bucket's ARN. `plan` and `apply` diff *desired* (your code) against *actual* (state) and call provider APIs to reconcile.

| Practice | Why |
|---|---|
| **Remote state** (S3 + DynamoDB lock, GCS, Terraform Cloud) | Your laptop ≠ source of truth |
| **Locking** | Two `apply`s at once corrupts state |
| **`terraform import`** | Bring an existing resource under management without recreating |
| **`terraform state rm`** | Forget about a resource (don't destroy it) |
| **Encrypt the bucket + restrict reads** | State holds secrets in plaintext for some resources |

```hcl
terraform {
  backend "s3" {
    bucket         = "my-team-tf-state"
    key            = "ml-platform/prod.tfstate"
    region         = "us-east-1"
    dynamodb_table = "tf-state-lock"
    encrypt        = true
  }
}
```

**Never commit `terraform.tfstate` to Git.** Always remote.

## 4 · Modules — reuse + composition

In [ ]:
module_call = '''# Use a public, battle-tested module from the Terraform registry
module "eks" {
  source  = "terraform-aws-modules/eks/aws"
  version = "~> 20.0"

  cluster_name    = var.cluster_name
  cluster_version = "1.30"

  vpc_id     = module.vpc.vpc_id
  subnet_ids = module.vpc.private_subnets

  eks_managed_node_groups = {
    cpu_general = {
      instance_types = ["m6a.large"]
      min_size = 2, max_size = 10, desired_size = 3
    }
    gpu_inference = {
      instance_types = ["g5.2xlarge"]            # 1× A10G GPU
      min_size = 0, max_size = 8, desired_size = 1
      labels = { workload = "gpu", accelerator = "nvidia-a10g" }
      taints = [{ key = "nvidia.com/gpu", value = "true", effect = "NO_SCHEDULE" }]
    }
  }
}
'''
print(module_call)

A **module** is just a directory of `.tf` files. You can write your own (`./modules/vllm-deployment/`) or pull battle-tested ones from the registry (`terraform-aws-modules/eks/aws`).

Notice how the GPU node group is **scaled from 0** to N — pair with the cluster autoscaler / Karpenter to spin GPUs up only when pods are pending. **Saves real money.**

## 5 · A real ML platform — assembling it

```
                ┌──────────── VPC ────────────┐
                │   public subnets / NAT      │
                │   private subnets (3 AZ)    │
                └─────────────┬───────────────┘
                              │
        ┌─────────────────────┼────────────────────┐
        ▼                     ▼                    ▼
   EKS cluster            S3 buckets           RDS Postgres
   ├ CPU node group       ├ models/            (pgvector)
   ├ GPU node group       ├ datasets/          ▲
   └ Karpenter            └ checkpoints/       │
        │                     ▲                │
        │                     │ IAM role for   │
        ▼                     │ service account│
   Helm releases ─────────────┘                │
   (M47) ──────────────► writes to S3 ─────────┘
```

```hcl
module "vpc" {
  source  = "terraform-aws-modules/vpc/aws"
  version = "~> 5.0"
  name    = "${var.cluster_name}-vpc"
  cidr    = "10.0.0.0/16"
  azs             = ["us-east-1a", "us-east-1b", "us-east-1c"]
  public_subnets  = ["10.0.1.0/24", "10.0.2.0/24", "10.0.3.0/24"]
  private_subnets = ["10.0.11.0/24","10.0.12.0/24","10.0.13.0/24"]
  enable_nat_gateway = true
}

resource "aws_s3_bucket" "models"      { bucket = "${var.cluster_name}-models" }
resource "aws_s3_bucket" "datasets"    { bucket = "${var.cluster_name}-datasets" }
resource "aws_s3_bucket" "checkpoints" { bucket = "${var.cluster_name}-checkpoints" }

# IAM role for service account (IRSA) — pods can read S3 without long-lived AWS keys
module "irsa_inference" {
  source  = "terraform-aws-modules/iam/aws//modules/iam-role-for-service-accounts-eks"
  version = "~> 5.0"
  role_name = "${var.cluster_name}-inference"
  oidc_providers = {
    main = {
      provider_arn               = module.eks.oidc_provider_arn
      namespace_service_accounts = ["llm:vllm-sa"]
    }
  }
  role_policy_arns = {
    s3 = aws_iam_policy.read_models.arn
  }
}
```

### A real platform also needs
- **Route 53 / DNS** records for your Ingress
- **ACM certificates** for TLS (cert-manager handles renewal — you bootstrap the issuer here)
- **CloudWatch / Stackdriver / Azure Monitor** log groups
- **Cost-allocation tags** on every resource (`team`, `env`, `project`)
- **GuardDuty / Security Hub** for compliance
- **VPC peering or Transit Gateway** if you have multi-VPC

## 6 · The workflow

```bash
$ terraform init       # download providers + connect to remote state
$ terraform fmt        # canonical formatting
$ terraform validate   # static check
$ terraform plan -out=plan.tfplan   # what would change?
$ terraform apply plan.tfplan       # apply that exact plan
$ terraform show       # current state
$ terraform destroy    # tear down (rare in prod; common in ephemeral envs)
```

Always **`plan` before `apply`**. Always **save the plan** so the apply matches what you reviewed.

For PR-driven workflows: **Atlantis** runs `plan` on every PR and posts the diff as a comment, runs `apply` on merge after approval. **Spacelift / env0 / Terraform Cloud** are managed alternatives.

## 7 · Workspaces, environments, and remote state

Three common patterns for splitting dev / staging / prod:

| Pattern | How |
|---|---|
| **Workspaces** (`terraform workspace`) | Same code, different state files. Quick & simple, but easy to mix up. |
| **Directory per env** | `envs/dev/`, `envs/staging/`, `envs/prod/` each with its own `main.tf` calling the same modules. **Most common in production.** |
| **Terragrunt** | thin wrapper over Terraform that DRYs up multi-env configs. Loved/hated. |

Production-grade layout:
```
infra/
├── modules/
│   ├── eks-cluster/
│   ├── ml-buckets/
│   └── observability/
└── envs/
    ├── dev/    (calls modules with dev values)
    ├── staging/
    └── prod/
```


## 8 · Secrets

| Don't | Do |
|---|---|
| `variable "db_password" { default = "..." }` in code | use **AWS Secrets Manager** / **GCP Secret Manager** / **Vault** + `data` lookups |
| Commit `*.tfvars` with secrets | encrypt with **SOPS**, or pass via env vars at apply time |
| Print secrets in `output` without `sensitive = true` | mark sensitive: `output "x" { value = ..., sensitive = true }` |
| Trust the state file because it's "internal" | encrypt the state bucket, restrict KMS access |

```hcl
data "aws_secretsmanager_secret_version" "db" {
  secret_id = "rds-prod-master"
}

resource "aws_db_instance" "pg" {
  password = jsondecode(data.aws_secretsmanager_secret_version.db.secret_string)["password"]
  ...
}
```


## 9 · Terraform vs Pulumi vs CDK vs Crossplane

| Tool | Strengths | Weaknesses |
|---|---|---|
| **Terraform / OpenTofu** | huge ecosystem, every cloud, mature state | HCL is limited; loops are awkward |
| **Pulumi** | real Python/TS — loops, types, tests | smaller community; same provider plugins |
| **AWS CDK / CDKTF** | concise; great defaults; AWS-first | leaky abstraction; CFN/TF bugs leak through |
| **Crossplane** | provision from inside k8s; GitOps-native | reconciliation lag; smaller ecosystem |
| **Ansible** (M49) | brilliant for configuring servers | wrong tool for cloud provisioning |

**One simple test.** "Does the operation create new infrastructure?" → IaC (Terraform). "Does it install software on existing infrastructure?" → Configuration management (Ansible / Helm).

## 10 · Day-2 + anti-patterns

| Topic | What to do |
|---|---|
| **Drift detection** | `terraform plan` on schedule (CI) — alerts if console edits diverged |
| **Module versioning** | pin `version = "~> 5.0"`; bump intentionally |
| **Atomic blast radius** | many small modules > one mega-stack; dev can't break prod |
| **Rotation** | rotate the state-bucket KMS key; rotate cloud creds |
| **Tags** | enforce `team`, `env`, `cost-center` via SCP/IAM policy |
| **Pre-commit hooks** | `terraform fmt`, `tflint`, `checkov` (security), `infracost` (cost) |

### Anti-patterns
- ❌ **One mega-`main.tf`** with 4 000 resources — every plan takes 30 minutes.
- ❌ **Shared state across teams** — accidental destroy across boundaries.
- ❌ **Mixing IaC and click-ops** — pick one. If you must edit by hand, *immediately* `terraform import` it back.
- ❌ **`terraform apply` from a laptop into prod.** Use a CI runner with scoped credentials.
- ❌ **Storing the state file in Git.** Plaintext secrets, conflict-merging nightmares, race conditions.


## ✅ Recap
- Terraform = declare cloud infrastructure in HCL; `plan` shows the diff, `apply` executes.
- 5 primitives: **provider · resource · data · variable · output**.
- **State is the central concept** — keep it remote, locked, encrypted.
- Modules for reuse; directory-per-env for dev/staging/prod.
- Pair with **Atlantis / Spacelift / Argo CD** for safe PR-driven applies.
- Terraform provisions infra; **Ansible (M49)** configures the servers/VMs that infra creates.

Next: **M49 — Ansible for ML Provisioning**.
